# Figure1c ucsf classification


In [ ]:
import gc
import os
import numpy as np
import pandas as pd
import anndata as ad
from tqdm import tqdm
from scipy import stats
from functools import partial
from datetime import datetime
import matplotlib.pyplot as plt
from multiprocessing import Pool
from joblib import dump, load
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, auc, precision_recall_curve
import warnings
warnings.filterwarnings('ignore')

In [ ]:
adata = ad.read_h5ad("data/adata_cohort1.h5ad")
adata.shape

In [ ]:
def donor_based_cv_split(X_train, y_train, adata, n_splits=5, random_state=42):

    train_donors = np.array(adata[adata.obs.index.isin(X_train.index)].obs.unique_patient_id.unique())

    np.random.seed(random_state)
    np.random.shuffle(train_donors)
    fold_size = len(train_donors) // n_splits
    folds = []

    for i in range(n_splits):
        if i == n_splits - 1:
            val_donors = train_donors[i*fold_size:]
        else:
            val_donors = train_donors[i*fold_size:(i+1)*fold_size]

        train_donors_fold = np.array([d for d in train_donors if d not in val_donors])

        train_indices = X_train.index[adata[adata.obs.index.isin(X_train.index)]
                                    .obs.unique_patient_id.isin(train_donors_fold)].tolist()
        val_indices = X_train.index[adata[adata.obs.index.isin(X_train.index)]
                                  .obs.unique_patient_id.isin(val_donors)].tolist()

        folds.append((train_indices, val_indices))

    return folds

In [ ]:
def analyze_single_iteration_with_roc(seed, adata, model_type='elastic_net',
                                     C=1.0, l1_ratio=0.5, n_splits=5):

    gc.collect()

    adjusted_adata_foldchange_df_filter = adata.to_df(layer="adjusted_fc_over_ag")
    X_filter = adjusted_adata_foldchange_df_filter.copy()
    HC_samples = np.array(adata[adata.obs.group==('healthy_control')].obs.index)
    y_filter = ~(X_filter.index.isin(HC_samples))
    y_filter = np.array(y_filter)
    X_filter[np.isnan(X_filter) | np.isinf(X_filter)] = 0

    n_pos = sum(y_filter)
    n_neg = len(y_filter) - n_pos
    class_ratio = n_pos / n_neg
    class_weights = {0: 1, 1: 1/class_ratio} if class_ratio < 1 else {0: class_ratio, 1: 1}

    cv_folds = donor_based_cv_split(X_filter, y_filter, adata,
                                   n_splits=n_splits, random_state=seed)

    iteration_results, models, roc_data = [], [], []

    for fold_idx, (train_idx, val_idx) in enumerate(cv_folds):

        X_fold_train = X_filter.loc[train_idx]
        X_fold_val = X_filter.loc[val_idx]
        train_mask = X_filter.index.isin(train_idx)
        val_mask = X_filter.index.isin(val_idx)
        y_fold_train = y_filter[train_mask]
        y_fold_val = y_filter[val_mask]

        if model_type == 'ridge':
            model = LogisticRegression(
                penalty='l2', solver='lbfgs', C=C,
                class_weight=class_weights, max_iter=500,
                random_state=seed + fold_idx, n_jobs=1)
        elif model_type == 'lasso':
            model = LogisticRegression(
                penalty='l1', solver='liblinear', C=C,
                class_weight=class_weights, max_iter=500,
                random_state=seed + fold_idx)
        elif model_type == 'elastic_net':
            model = LogisticRegression(
                penalty='elasticnet', solver='saga',
                l1_ratio=l1_ratio, C=C,
                class_weight=class_weights,
                max_iter=100, tol=1e-3,
                random_state=seed + fold_idx,
                n_jobs=1)

        model.fit(X_fold_train, y_fold_train)
        y_pred_proba = model.predict_proba(X_fold_val)[:, 1]

        fpr, tpr, thresholds = roc_curve(y_fold_val, y_pred_proba)
        prec, rec, pr_thresholds = precision_recall_curve(y_fold_val, y_pred_proba)

        iteration_results.append({
            'seed': seed, 'fold': fold_idx,
            'auroc': roc_auc_score(y_fold_val, y_pred_proba),
            'auprc': auc(rec, prec),
            'n_features': np.sum(model.coef_[0] != 0),
            'n_train_samples': len(X_fold_train),
            'n_test_samples': len(X_fold_val),
            'n_train_donors': len(set(adata[adata.obs.index.isin(train_idx)].obs.unique_patient_id)),
            'n_test_donors': len(set(adata[adata.obs.index.isin(val_idx)].obs.unique_patient_id))
        })

        models.append(model)

        roc_data.append({
            'seed': seed, 'fold': fold_idx, 'fpr': fpr, 'tpr': tpr,
            'thresholds': thresholds, 'y_true': y_fold_val,
            'y_pred_proba': y_pred_proba})

    feature_names = list(X_filter.columns)

    del X_filter
    gc.collect()

    return iteration_results, models, roc_data, feature_names

def run_batched_cv_analysis(adata, output_dir, model_type='elastic_net',
                           C=1.0, l1_ratio=0.5, n_iterations=20,
                           batch_size=5, n_jobs=5):
    os.makedirs(output_dir, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"Model Configuration:")
    print(f"Type: {model_type}")
    print(f"C: {C}")
    if model_type == 'elastic_net':
        print(f"l1_ratio: {l1_ratio}")
    print(f"Iterations: {n_iterations}, Batch size: {batch_size}, Jobs: {n_jobs}")
    print(f"Output directory: {output_dir}")
    print(f"{'='*60}\n")

    all_results = []
    all_models = {}
    all_roc_data = []
    feature_names = None

    n_batches = (n_iterations + batch_size - 1) // batch_size

    for batch in tqdm(range(n_batches), desc="Processing batches"):

        start_seed = batch * batch_size
        end_seed = min((batch + 1) * batch_size, n_iterations)
        seeds = [42 + i * 1000 for i in range(start_seed, end_seed)]

        print(f"\nBatch {batch+1}/{n_batches}: Processing iterations {start_seed}-{end_seed-1}")

        analyze_func = partial(
            analyze_single_iteration_with_roc,
            adata=adata, model_type=model_type,
            C=C, l1_ratio=l1_ratio, n_splits=5)

        with Pool(n_jobs) as pool:
            batch_results = list(pool.imap(analyze_func, seeds))

        for iteration_idx, (seed, results) in enumerate(zip(seeds, batch_results)):
            iter_results, models, roc_data, features = results

            if feature_names is None:
                feature_names = features

            all_results.extend(iter_results)

            for fold_idx, model in enumerate(models):
                all_models[f"seed_{seed}_fold_{fold_idx}"] = model

            all_roc_data.extend(roc_data)

        temp_df = pd.DataFrame(all_results)
        temp_df.to_csv(f'{output_dir}/results_temp.csv', index=False)

        dump(all_models, f'{output_dir}/models_batch_{batch+1}.joblib')
        dump(all_roc_data, f'{output_dir}/roc_data_batch_{batch+1}.joblib')
        gc.collect()

    results_df = pd.DataFrame(all_results)

    print("\nSaving final results...")
    results_df.to_csv(os.path.join(output_dir, 'results.csv'), index=False)
    dump(all_models, os.path.join(output_dir, 'models.joblib'))
    dump(all_roc_data, os.path.join(output_dir, 'roc_data.joblib'))

    with open(os.path.join(output_dir, 'feature_names.txt'), 'w') as f:
        f.write('\n'.join(feature_names))

    config = {
        'model_type': model_type, 'C': C,
        'l1_ratio': l1_ratio if model_type == 'elastic_net' else None,
        'n_iterations': n_iterations, 'n_splits': 5,
        'batch_size': batch_size, 'n_jobs': n_jobs,
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
    dump(config, os.path.join(output_dir, 'config.joblib'))

    if os.path.exists(f'{output_dir}/results_temp.csv'):
        os.remove(f'{output_dir}/results_temp.csv')
    for batch in range(n_batches):
        for file_type in ['models', 'roc_data']:
            batch_file = f'{output_dir}/{file_type}_batch_{batch+1}.joblib'
            if os.path.exists(batch_file):
                os.remove(batch_file)

    print(f"\nResults Summary:")
    print(f"AUROC: {results_df['auroc'].mean():.4f} ± {results_df['auroc'].std():.4f}")
    print(f"AUPRC: {results_df['auprc'].mean():.4f} ± {results_df['auprc'].std():.4f}")
    print(f"Features: {results_df['n_features'].mean():.0f} ± {results_df['n_features'].std():.0f}")
    print(f"Min features: {results_df['n_features'].min()}")
    print(f"Max features: {results_df['n_features'].max()}")

    return {
        'results_df': results_df, 'models': all_models,
        'roc_data': all_roc_data, 'feature_names': feature_names,
        'config': config}

In [ ]:
results4 = run_batched_cv_analysis(
    adata,
    output_dir='results/classification_sle_ucsf/results_elasticnet_C1_l1_0.5',
    model_type='elastic_net', C=1, l1_ratio=0.5,
    n_iterations=20, batch_size=20,
    n_jobs=10)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import load
from scipy import stats

def find_optimal_threshold(mean_fpr, mean_tpr):
    youden_index = mean_tpr - mean_fpr
    optimal_idx = np.argmax(youden_index)
    return optimal_idx, mean_fpr[optimal_idx], mean_tpr[optimal_idx]

def find_closest_to_01(mean_fpr, mean_tpr):
    distances = np.sqrt(mean_fpr**2 + (1 - mean_tpr)**2)
    optimal_idx = np.argmin(distances)
    return optimal_idx, mean_fpr[optimal_idx], mean_tpr[optimal_idx]

def find_sensitivity_at_specificity(mean_fpr, mean_tpr, target_specificity=0.90):
    target_fpr = 1 - target_specificity
    idx = np.argmin(np.abs(mean_fpr - target_fpr))
    return idx, mean_fpr[idx], mean_tpr[idx]

def find_specificity_at_sensitivity(mean_fpr, mean_tpr, target_sensitivity=0.90):
    idx = np.argmin(np.abs(mean_tpr - target_sensitivity))
    return idx, mean_fpr[idx], mean_tpr[idx]

def plot_roc_publication(
    primary_output_dir,
    comparison_output_dir=None,
    save_plot=True,
    output_dir=None,

    show_confidence_interval=True,
    ci_method='percentile',
    ci_level=0.95,
    show_individual_curves=False,

    show_operating_point=True,
    operating_point_type='sens_at_spec',
    target_specificity=0.90,
    target_sensitivity=0.90,

    primary_label=None,
    comparison_label=None,

    figsize=(7, 7),
    primary_color="#b1615c",
    comparison_color="#c9c9dd",

    show_auc_ci_in_legend=True,
    decimal_places=2,
):

    plt.rcParams['mathtext.fontset'] = 'custom'

    plt.rcParams['axes.linewidth'] = 1.2
    plt.rcParams['xtick.major.width'] = 1.2
    plt.rcParams['ytick.major.width'] = 1.2

    results_df_primary = pd.read_csv(f'{primary_output_dir}/results.csv')
    roc_data_primary = load(f'{primary_output_dir}/roc_data.joblib')
    config_primary = load(f'{primary_output_dir}/config.joblib')

    stats_dict = {'primary': {}, 'comparison': None}

    fig, ax = plt.subplots(figsize=figsize)

    mean_fpr = np.linspace(0, 1, 1000)

    tprs_primary = []
    for data in roc_data_primary:
        interp_tpr = np.interp(mean_fpr, data['fpr'], data['tpr'])
        interp_tpr[0] = 0.0
        tprs_primary.append(interp_tpr)
    tprs_primary = np.array(tprs_primary)

    mean_tpr_primary = np.mean(tprs_primary, axis=0)
    mean_tpr_primary[-1] = 1.0

    aucs_primary = results_df_primary['auroc'].values
    mean_auc_primary = np.mean(aucs_primary)
    std_auc_primary = np.std(aucs_primary)
    ci_low_auc_primary = np.percentile(aucs_primary, (1 - ci_level) / 2 * 100)
    ci_high_auc_primary = np.percentile(aucs_primary, (1 + ci_level) / 2 * 100)

    stats_dict['primary'] = {
        'mean_auc': mean_auc_primary,
        'std_auc': std_auc_primary,
        'ci_low_auc': ci_low_auc_primary,
        'ci_high_auc': ci_high_auc_primary,
        'n_iterations': len(aucs_primary),
    }

    if show_individual_curves:
        for i in range(len(tprs_primary)):
            ax.plot(mean_fpr, tprs_primary[i], alpha=0.08, color=primary_color,
                    linewidth=0.5, zorder=1)

    if ci_method == 'percentile':
        alpha_low = (1 - ci_level) / 2 * 100
        alpha_high = (1 + ci_level) / 2 * 100
        tprs_lower = np.percentile(tprs_primary, alpha_low, axis=0)
        tprs_upper = np.percentile(tprs_primary, alpha_high, axis=0)
        ci_label = f'{int(ci_level*100)}% CI'
    else:
        std_tpr = np.std(tprs_primary, axis=0)
        tprs_lower = np.maximum(mean_tpr_primary - std_tpr, 0)
        tprs_upper = np.minimum(mean_tpr_primary + std_tpr, 1)
        ci_label = '± 1 s.d.'

    if show_confidence_interval:
        ax.fill_between(mean_fpr, tprs_lower, tprs_upper,
                        color=primary_color, alpha=0.15, zorder=2,
                        label=ci_label, linewidth=0)

    if primary_label is None:
        model_type = config_primary.get('model_type', 'Model')
        if model_type == 'elastic_net':
            model_name = 'Elastic Net'
        elif model_type == 'lasso':
            model_name = 'LASSO'
        elif model_type == 'ridge':
            model_name = 'Ridge'
        else:
            model_name = model_type.title()
    else:
        model_name = primary_label

    fmt = f'.{decimal_places}f'
    if show_auc_ci_in_legend:
        auc_str = f'AUC = {mean_auc_primary:{fmt}} ({ci_low_auc_primary:{fmt}}–{ci_high_auc_primary:{fmt}})'
    else:
        auc_str = f'AUC = {mean_auc_primary:{fmt}} ± {std_auc_primary:{fmt}}'

    primary_full_label = f'{model_name}: {auc_str}'

    ax.plot(mean_fpr, mean_tpr_primary, color=primary_color, linewidth=2.5,
            label=primary_full_label, zorder=4)

    if show_operating_point:
        if operating_point_type == 'sens_at_spec':
            op_idx, op_fpr, op_tpr = find_sensitivity_at_specificity(
                mean_fpr, mean_tpr_primary, target_specificity)
            op_label = f'Sens = {op_tpr:{fmt}} at {int(target_specificity*100)}% Spec'
        elif operating_point_type == 'spec_at_sens':
            op_idx, op_fpr, op_tpr = find_specificity_at_sensitivity(
                mean_fpr, mean_tpr_primary, target_sensitivity)
            op_label = f'Spec = {1-op_fpr:{fmt}} at {int(target_sensitivity*100)}% Sens'
        elif operating_point_type == 'closest_to_01':
            op_idx, op_fpr, op_tpr = find_closest_to_01(mean_fpr, mean_tpr_primary)
            op_label = f'Closest to (0,1): Sens = {op_tpr:{fmt}}, Spec = {1-op_fpr:{fmt}}'
        else:
            op_idx, op_fpr, op_tpr = find_optimal_threshold(mean_fpr, mean_tpr_primary)
            op_label = f'Youden: Sens = {op_tpr:{fmt}}, Spec = {1-op_fpr:{fmt}}'

        ax.scatter([op_fpr], [op_tpr], color=primary_color, s=70, zorder=6,
                   edgecolors='white', linewidths=1.5, marker='o')

        if op_tpr > 0.5:
            text_offset = (-0.12, -0.08)
            ha = 'left'
        else:
            text_offset = (0.05, 0.05)
            ha = 'left'

        ax.annotate(op_label,
                    xy=(op_fpr, op_tpr),
                    xytext=(op_fpr + text_offset[0], op_tpr + text_offset[1]),
                    fontsize=10,
                    ha=ha,
                    arrowprops=dict(arrowstyle='->', color='gray', lw=0.8,
                                    connectionstyle='arc3,rad=0.1'),
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                              edgecolor='gray', alpha=0.9),
                    zorder=7)

        stats_dict['primary']['operating_point'] = {
            'type': operating_point_type,
            'fpr': op_fpr,
            'tpr': op_tpr,
            'sensitivity': op_tpr,
            'specificity': 1 - op_fpr,
        }

    if comparison_output_dir:
        results_df_comp = pd.read_csv(f'{comparison_output_dir}/results.csv')
        roc_data_comp = load(f'{comparison_output_dir}/roc_data.joblib')
        config_comp = load(f'{comparison_output_dir}/config.joblib')

        tprs_comp = []
        for data in roc_data_comp:
            interp_tpr = np.interp(mean_fpr, data['fpr'], data['tpr'])
            interp_tpr[0] = 0.0
            tprs_comp.append(interp_tpr)
        tprs_comp = np.array(tprs_comp)

        mean_tpr_comp = np.mean(tprs_comp, axis=0)
        mean_tpr_comp[-1] = 1.0

        aucs_comp = results_df_comp['auroc'].values
        mean_auc_comp = np.mean(aucs_comp)
        std_auc_comp = np.std(aucs_comp)
        ci_low_auc_comp = np.percentile(aucs_comp, (1 - ci_level) / 2 * 100)
        ci_high_auc_comp = np.percentile(aucs_comp, (1 + ci_level) / 2 * 100)

        stats_dict['comparison'] = {
            'mean_auc': mean_auc_comp,
            'std_auc': std_auc_comp,
            'ci_low_auc': ci_low_auc_comp,
            'ci_high_auc': ci_high_auc_comp,
            'n_iterations': len(aucs_comp),
        }

        if comparison_label is None:
            model_type_comp = config_comp.get('model_type', 'Model')
            if model_type_comp == 'elastic_net':
                comp_name = 'Elastic Net'
            elif model_type_comp == 'lasso':
                comp_name = 'LASSO'
            elif model_type_comp == 'ridge':
                comp_name = 'Ridge'
            else:
                comp_name = model_type_comp.title()
        else:
            comp_name = comparison_label

        if show_auc_ci_in_legend:
            auc_str_comp = f'AUC = {mean_auc_comp:{fmt}} ({ci_low_auc_comp:{fmt}}–{ci_high_auc_comp:{fmt}})'
        else:
            auc_str_comp = f'AUC = {mean_auc_comp:{fmt}} ± {std_auc_comp:{fmt}}'

        comp_full_label = f'{comp_name}: {auc_str_comp}'

        ax.plot(mean_fpr, mean_tpr_comp, color=comparison_color, linewidth=2.5,
                linestyle='--', label=comp_full_label, zorder=3)

        t_stat, p_value = stats.ttest_rel(aucs_primary, aucs_comp)
        stats_dict['comparison']['t_stat'] = t_stat
        stats_dict['comparison']['p_value'] = p_value

    ax.plot([0, 1], [0, 1], color='gray', linewidth=1.5, linestyle='--',
            alpha=0.7, label='Random classifier', zorder=1)

    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])
    ax.set_xlabel('1 − Specificity', fontsize=14, fontweight='medium')
    ax.set_ylabel('Sensitivity', fontsize=14, fontweight='medium')
    ax.tick_params(axis='both', which='major', labelsize=12, length=5)

    ax.set_xticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])

    ax.grid(True, alpha=0.2, linestyle='-', linewidth=0.5, zorder=0)

    ax.set_aspect('equal', adjustable='box')

    legend = ax.legend(loc='lower right', fontsize=11, framealpha=0.95,
                       edgecolor='gray', fancybox=False)
    legend.get_frame().set_linewidth(0.8)

    plt.tight_layout()

    if save_plot:
        save_dir = output_dir if output_dir else primary_output_dir
        save_name = 'roc_curve_comparison' if comparison_output_dir else 'roc_curve'

        plt.savefig(f'{save_dir}/{save_name}.pdf', bbox_inches='tight',
                    facecolor='white', edgecolor='none')

    print("\n" + "="*60)
    print("ROC CURVE STATISTICS")
    print("="*60)

    n_folds = config_primary.get('n_splits', 5)
    n_repeats = len(aucs_primary) // n_folds
    print(f"\nValidation: {n_folds}-fold CV × {n_repeats} repeats = {len(aucs_primary)} iterations")

    print(f"\nPrimary Model ({model_name}):")
    print(f"  Mean AUC: {mean_auc_primary:.4f}")
    print(f"  Std AUC:  {std_auc_primary:.4f}")
    print(f"  95% CI:   [{ci_low_auc_primary:.4f}, {ci_high_auc_primary:.4f}]")

    if show_operating_point:
        print(f"\n  Operating Point ({operating_point_type}):")
        print(f"    Sensitivity: {stats_dict['primary']['operating_point']['sensitivity']:.4f}")
        print(f"    Specificity: {stats_dict['primary']['operating_point']['specificity']:.4f}")

    if comparison_output_dir:
        print(f"\nComparison Model ({comp_name}):")
        print(f"  Mean AUC: {mean_auc_comp:.4f}")
        print(f"  Std AUC:  {std_auc_comp:.4f}")
        print(f"  95% CI:   [{ci_low_auc_comp:.4f}, {ci_high_auc_comp:.4f}]")

        print(f"\nStatistical Comparison (paired t-test):")
        print(f"  t-statistic: {t_stat:.4f}")
        print(f"  p-value:     {p_value:.4f}")
        if p_value < 0.001:
            print("  Significance: *** (p < 0.001)")
        elif p_value < 0.01:
            print("  Significance: ** (p < 0.01)")
        elif p_value < 0.05:
            print("  Significance: * (p < 0.05)")
        else:
            print("  Significance: n.s. (p ≥ 0.05)")

    print("\n" + "="*60)

    plt.show()

    return fig, ax, stats_dict

In [ ]:
fig3, ax3, stats3 = plot_roc_publication(
        primary_output_dir='results/classification_sle_ucsf/results_elasticnet_C1_l1_0.5',
        save_plot=True,
        show_confidence_interval=True,
        ci_method='percentile',
        ci_level=0.95,
        show_operating_point=True,
        operating_point_type='closest_to_01',
        primary_label='PhIP-seq signature',
        comparison_label='LASSO baseline',
        figsize=(6, 6),
        decimal_places=2,
        show_auc_ci_in_legend=True,
    )